In [1]:
import os
import torch
from torch.utils.data import DataLoader
from pathlib import Path
import sys

project_root = Path(os.getcwd()).parent
print(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.preprocessing.pipeline import Pipeline
from src.data.datasets.universal_dataset import CVADataset
from src.models.network import DiffusionAttn
from src.models.diffusion import GaussianDiffusion
from src.train.trainer import setup_optimizer, DiffusionTrainer

/mnt/c/Users/edtop/ITMO/THESIS_CV_DIFF


/mnt/c/Users/edtop/ITMO/THESIS_CV_DIFF/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Global hyperpar
EPOCHS = 100
BATCH_SIZE = 64
LR = 0.00095
WEIGHT_DECAY = 0.01
TIMESTEPS = 1000 

TEST_INHIBITOR = "2-mercaptobenzimidazole" 

NUM_CYCLE = [1, 2, 3, 4]
save_dir = project_root / "experiments" / "run_01"
SAVE_DIR = str(save_dir)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Device: {DEVICE}")

[*] Device: cuda


In [3]:
pipe = Pipeline(
    num_cycle=NUM_CYCLE, 
    test_inhibitor=TEST_INHIBITOR, 
    norm_feat=True, 
    use_wavelet=False
)

train_dataset = CVADataset(
    vol=pipe.train_voltage,
    cur=pipe.train_current,
    desc_df=pipe.train_analyzed_data
)

val_dataset = CVADataset(
    vol=pipe.test_voltage,
    cur=pipe.test_current,
    desc_df=pipe.test_analyzed_data
)

In [4]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"Size Train: {len(train_dataset)} samples")
print(f"Size Val: {len(val_dataset)} samples")

Size Train: 2684 samples
Size Val: 776 samples


In [5]:
num_desc_features = train_dataset[0]["features"].shape[0]

net = DiffusionAttn(
        in_channels=2, 
        desc_features=num_desc_features, 
        base_channels=64
    )
    
diffusion = GaussianDiffusion(model=net, timesteps=TIMESTEPS)

optimizer, scheduler = setup_optimizer(
    model=net, 
    lr=LR, 
    weight_decay=WEIGHT_DECAY, 
    epochs=EPOCHS
)

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.1.2+cu121
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [6]:
net

DiffusionAttn(
  (signal_head): SignalHead(
    (proj): Conv1d(2, 64, kernel_size=(7,), stride=(1,), padding=(3,))
  )
  (desc_head): DescriptorHead(
    (silu_mlp): Sequential(
      (0): Linear(in_features=41, out_features=256, bias=True)
      (1): SiLU()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): SiLU()
    )
  )
  (time_head): TimeEmbedding(
    (mlp): Sequential(
      (0): SinusoidalPositionEmbeddings()
      (1): Linear(in_features=64, out_features=256, bias=True)
      (2): SiLU()
      (3): Linear(in_features=256, out_features=256, bias=True)
    )
  )
  (encoder): Encoder(
    (down1): ResnetBlock1D(
      (norm1): GroupNorm(8, 64, eps=1e-05, affine=True)
      (conv1): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
      (norm2): GroupNorm(8, 64, eps=1e-05, affine=True)
      (dropout): Dropout(p=0.2, inplace=False)
      (conv2): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
      (cond_proj): Sequential(
        (

In [6]:
trainer = DiffusionTrainer(
        diffusion_model=diffusion,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        device=DEVICE,
        save_dir=SAVE_DIR, 
        vol_scaler=pipe.vol_scaler,
        cur_scaler=pipe.cur_scaler
    )

print("\n" + "="*40)
print("Start")
print("="*40)
trainer.fit(epochs=EPOCHS)


Start
Teaching on cuda...


Epoch 1 [Val]: 100%|██████████| 13/13 [00:02<00:00,  5.38it/s, val_loss=0.1880]


Epoch 1 | Train Loss: 0.5749 | Val Loss: 0.3898 | LR: 0.000950
Saved best model (Val Loss: 0.3898)


Sampling: 100%|██████████| 1000/1000 [00:49<00:00, 20.34it/s]


Epoch 2 | Train Loss: 0.1499 | Val Loss: 0.2490 | LR: 0.000949
Saved best model (Val Loss: 0.2490)


...

Epoch 37 | Train Loss: 0.0197 | Val Loss: 0.0542 | LR: 0.000664


Sampling: 100%|██████████| 1000/1000 [00:45<00:00, 21.78it/s]


Epoch 38 | Train Loss: 0.0198 | Val Loss: 0.0493 | LR: 0.000650


Epoch 39 [Train]:  68%|██████▊   | 28/41 [00:27<00:12,  1.00it/s, loss=0.0134]


KeyboardInterrupt: 